---
format:
  html:
    code-fold: true
jupyter: python3
---

### **Cell 1: Task Definition and Plan**

**Tiny model**

- I will use `distilgpt2` from Hugging Face Transformers.
- It is a small, pretrained decoder-only causal language model that fits on a typical GPU/CPU.

**Toy task**

- Task: single-step integer addition of two one-digit numbers.
- Prompt template:  
  `Add the numbers: {a} + {b} =`
- Desired output: the correct sum as a plain decimal string.  
  Example:  
  - Prompt: `Add the numbers: 7 + 5 =`  
  - Target output: `12`

**Reward logic**

- Reward is rule-based and automatically computed.
- For each `(prompt, response)` pair:
  - Extract the first integer that appears in the model’s response.
  - Compare it with the ground-truth sum for that prompt.
  - Reward = `1.0` if the extracted integer equals the correct sum; otherwise `0.0`.
- I will also add a small sanity-check routine that prints  
  `(prompt, response, parsed_answer, target, reward)` for a few examples.

**GRPO setup**

- Datasets:
  - Training set: 1,000 randomly generated addition problems.
  - Evaluation set: 200 held-out addition problems from the same distribution.
- Group size: `G = 4` sampled completions per prompt during GRPO updates.
- Advantage computation: for each prompt, normalize rewards within its group:  

  $$A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r) + \epsilon}$$

- Policy loss:
  - Use the negative advantage-weighted log-probability of the generated tokens.
  - Optionally add a KL penalty term against a frozen copy of the initial model to keep the policy close to the reference.

**Hypothesis**

- Before fine-tuning, the base model will sometimes produce correct sums but with relatively low average reward.
- After GRPO fine-tuning:
  - The mean reward on the evaluation set should increase.
  - The model’s responses should more consistently contain the correct sum for the given prompt.
  - With a mild KL penalty, the model should improve on the addition task without obvious reward hacking
    (e.g., spamming random numbers).

In [1]:
# Cell 2: Setup and Model Loading

import math
import random
from typing import List, Tuple

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# If you get import errors, install the packages manually, e.g.:
# !pip install transformers torch

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Tiny model name (matches Cell 1 description)
MODEL_NAME = "distilgpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 family has no pad_token by default; align pad with eos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))  # in case pad token was added
model.config.pad_token_id = tokenizer.pad_token_id

model.to(device)
model.train()

# Print a short summary to verify loading
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Loaded model: {MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded model: distilgpt2
Total parameters: 81,912,576
Trainable parameters: 81,912,576


In [2]:
# Cell 3: Dataset Generation

import pprint

def generate_addition_dataset(
    n_samples: int,
    min_val: int = 0,
    max_val: int = 9,
) -> list:
    """
    Generate a toy dataset for single-step integer addition.

    Each example is a dict with:
      - 'prompt': the text shown to the model
      - 'a', 'b': the operands
      - 'target': the correct sum (integer)
    """
    data = []
    for _ in range(n_samples):
        a = random.randint(min_val, max_val)
        b = random.randint(min_val, max_val)
        prompt = f"Add the numbers: {a} + {b} ="
        target = a + b
        data.append(
            {
                "prompt": prompt,
                "a": a,
                "b": b,
                "target": target,
            }
        )
    return data


# Generate train / eval splits
N_TRAIN = 1000
N_EVAL = 200

train_data = generate_addition_dataset(N_TRAIN)
eval_data = generate_addition_dataset(N_EVAL)

print(f"Train size: {len(train_data)}")
print(f"Eval size:  {len(eval_data)}\n")

print("A few training examples:")
pprint.pp(train_data[:5])

Train size: 1000
Eval size:  200

A few training examples:
[{'prompt': 'Add the numbers: 1 + 0 =', 'a': 1, 'b': 0, 'target': 1},
 {'prompt': 'Add the numbers: 4 + 3 =', 'a': 4, 'b': 3, 'target': 7},
 {'prompt': 'Add the numbers: 3 + 2 =', 'a': 3, 'b': 2, 'target': 5},
 {'prompt': 'Add the numbers: 1 + 8 =', 'a': 1, 'b': 8, 'target': 9},
 {'prompt': 'Add the numbers: 1 + 9 =', 'a': 1, 'b': 9, 'target': 10}]


In [3]:
# Cell 4: Reward Function Implementation

import re

def extract_addition_target_from_prompt(prompt: str) -> int:
    """
    Parse 'a + b' from a prompt like:
        'Add the numbers: 3 + 5 ='
    and return a + b as an integer.
    """
    match = re.search(r"(-?\d+)\s*\+\s*(-?\d+)", prompt)
    if match is None:
        raise ValueError(f"Could not parse numbers from prompt: {prompt}")
    a = int(match.group(1))
    b = int(match.group(2))
    return a + b


def extract_first_int_from_text(text: str):
    """
    Return the first integer found in the text, or None if none is found.
    """
    match = re.search(r"-?\d+", text)
    if match is None:
        return None
    return int(match.group(0))


def get_reward(prompts, responses) -> torch.Tensor:
    """
    Compute rewards for a batch of (prompt, response) pairs.

    Reward definition (single-step addition task):
      - 1.0 if the first integer in the response equals the correct sum
        implied by the prompt (a + b).
      - 0.0 otherwise.

    Returns:
      torch.Tensor of shape [batch_size] on the same device as the model.
    """
    assert len(prompts) == len(responses), "prompts and responses must have same length"

    rewards = torch.zeros(len(prompts), dtype=torch.float32, device=device)
    for i, (p, r) in enumerate(zip(prompts, responses)):
        target = extract_addition_target_from_prompt(p)
        pred = extract_first_int_from_text(r)
        reward = 1.0 if (pred is not None and pred == target) else 0.0
        rewards[i] = reward
    return rewards


# -------- Sanity Check --------

manual_prompts = [
    "Add the numbers: 1 + 2 =",
    "Add the numbers: 7 + 5 =",
    "Add the numbers: 9 + 9 =",
]

good_responses = [
    "3",
    "The answer is 12.",
    "Result: 18! 🎉",
]

bad_responses = [
    "4",
    "I think it is 10.",
    "Result: 19!",
]

print("=== Sanity check: good responses ===")
good_rewards = get_reward(manual_prompts, good_responses)
for p, r, rew in zip(manual_prompts, good_responses, good_rewards.tolist()):
    target = extract_addition_target_from_prompt(p)
    pred = extract_first_int_from_text(r)
    print(f"Prompt:   {p}")
    print(f"Response: {r}")
    print(f"Target / Parsed: {target} / {pred}")
    print(f"Reward:  {rew}\n")

print("=== Sanity check: bad responses ===")
bad_rewards = get_reward(manual_prompts, bad_responses)
for p, r, rew in zip(manual_prompts, bad_responses, bad_rewards.tolist()):
    target = extract_addition_target_from_prompt(p)
    pred = extract_first_int_from_text(r)
    print(f"Prompt:   {p}")
    print(f"Response: {r}")
    print(f"Target / Parsed: {target} / {pred}")
    print(f"Reward:  {rew}\n")

# -------- Extra edge-case checks --------

edge_prompts = [
    "Add the numbers: 2 + 3 =",
    "Add the numbers: 2 + 3 =",
]

edge_responses = [
    "No idea.",              # no integer -> reward 0.0
    "The answer is 5 6",     # multiple integers, first is correct -> reward 1.0
]

print("=== Sanity check: edge cases ===")
edge_rewards = get_reward(edge_prompts, edge_responses)
for p, r, rew in zip(edge_prompts, edge_responses, edge_rewards.tolist()):
    target = extract_addition_target_from_prompt(p)
    pred = extract_first_int_from_text(r)
    print(f"Prompt:   {p}")
    print(f"Response: {r!r}")
    print(f"Target / Parsed: {target} / {pred}")
    print(f"Reward:  {rew}\n")


=== Sanity check: good responses ===
Prompt:   Add the numbers: 1 + 2 =
Response: 3
Target / Parsed: 3 / 3
Reward:  1.0

Prompt:   Add the numbers: 7 + 5 =
Response: The answer is 12.
Target / Parsed: 12 / 12
Reward:  1.0

Prompt:   Add the numbers: 9 + 9 =
Response: Result: 18! 🎉
Target / Parsed: 18 / 18
Reward:  1.0

=== Sanity check: bad responses ===
Prompt:   Add the numbers: 1 + 2 =
Response: 4
Target / Parsed: 3 / 4
Reward:  0.0

Prompt:   Add the numbers: 7 + 5 =
Response: I think it is 10.
Target / Parsed: 12 / 10
Reward:  0.0

Prompt:   Add the numbers: 9 + 9 =
Response: Result: 19!
Target / Parsed: 18 / 19
Reward:  0.0

=== Sanity check: edge cases ===
Prompt:   Add the numbers: 2 + 3 =
Response: 'No idea.'
Target / Parsed: 5 / None
Reward:  0.0

Prompt:   Add the numbers: 2 + 3 =
Response: 'The answer is 5 6'
Target / Parsed: 5 / 5
Reward:  1.0



In [4]:
# Cell 5: The GRPO Step

import torch.nn.functional as F

# GRPO hyperparameters
GROUP_SIZE = 4          # G ≥ 2
MAX_NEW_TOKENS = 5
TEMPERATURE = 1.0
TOP_K = 50
KL_COEF = 0.0           # set > 0.0 if you want KL regularization

# Frozen reference model for optional KL regularization and baseline comparison
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
ref_model.resize_token_embeddings(len(tokenizer))
ref_model.config.pad_token_id = tokenizer.pad_token_id
ref_model.to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)


def grpo_step(
    prompts,
    group_size: int = GROUP_SIZE,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
    kl_coef: float = KL_COEF,
):
    """
    Perform a single GRPO update step (no optimizer step inside).

    Args:
        prompts: list of prompt strings (batch of x).
        group_size: number of sampled completions per prompt (G).
        max_new_tokens: maximum length of each completion.
        temperature, top_k: sampling parameters.
        kl_coef: coefficient for optional KL penalty.

    Returns:
        dict with:
            - loss: scalar tensor (policy loss [+ KL]).
            - advantages: tensor of shape [batch_size * group_size].
            - rewards: tensor of shape [batch_size * group_size].
            - logprobs: tensor of shape [batch_size * group_size].
            - responses: list of decoded completions.
            - prompts: list of prompts repeated group_size times.
    """
    model.train()
    batch_size = len(prompts)
    assert batch_size > 0

    # ----- 1. Encode prompts -----
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    input_ids = enc["input_ids"]          # [B, T_in]
    attention_mask = enc["attention_mask"]
    prompt_lengths = attention_mask.sum(dim=1)  # [B]

    # ----- 2. Generation: G completions per prompt -----
    # generate returns [B * G, T_out]
    gen_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        max_new_tokens=max_new_tokens,
        num_return_sequences=group_size,
        pad_token_id=tokenizer.pad_token_id,
    )

    sequences = gen_sequences  # [B * G, T]
    n_sequences, T = sequences.shape

    # Valid lengths (excluding padding)
    non_pad_mask = sequences != tokenizer.pad_token_id
    seq_lens = non_pad_mask.sum(dim=1)  # [B * G]

    # For bookkeeping: which prompt each sequence came from
    expanded_prompt_lengths = prompt_lengths.repeat_interleave(group_size)  # [B * G]

    # ----- 3. Compute log-probabilities under current model -----
    outputs = model(sequences, attention_mask=non_pad_mask)
    logits = outputs.logits  # [B * G, T, V]
    log_probs_full = F.log_softmax(logits, dim=-1)

    # logprob of each token (except the very first) under current model
    # token_logprobs[i, j] = log p(token at position j+1 | up to j)
    token_logprobs = log_probs_full[:, :-1, :].gather(
        dim=-1,
        index=sequences[:, 1:].unsqueeze(-1),
    ).squeeze(-1)  # [B * G, T - 1]

    # Sum log-probs over ONLY the generated tokens (ignore prompt tokens)
    seq_logprobs = []
    for i in range(n_sequences):
        L_prompt = int(expanded_prompt_lengths[i].item())   # number of prompt tokens
        T_i = int(seq_lens[i].item())                       # number of non-pad tokens

        if T_i <= L_prompt:
            # No generated tokens (unlikely with max_new_tokens > 0); zero contribution
            lp = token_logprobs[i, :0].sum()
        else:
            # Generated tokens are positions [L_prompt, ..., T_i-1] in 'sequences'
            # Their logprobs live at indices [L_prompt-1, ..., T_i-2] in 'token_logprobs'
            start = L_prompt - 1
            end = T_i - 1
            lp = token_logprobs[i, start:end].sum()
        seq_logprobs.append(lp)

    seq_logprobs = torch.stack(seq_logprobs, dim=0)  # [B * G]

    # ----- 4. Decode responses and compute rewards -----
    expanded_prompts = [p for p in prompts for _ in range(group_size)]
    responses = []

    for i in range(n_sequences):
        L_prompt = int(expanded_prompt_lengths[i].item())
        T_i = int(seq_lens[i].item())
        # Take only the generated continuation (exclude prompt)
        gen_tokens = sequences[i, L_prompt:T_i]
        resp_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        responses.append(resp_text.strip())

    rewards = get_reward(expanded_prompts, responses)  # [B * G], on device

    # ----- 5. Compute group-wise normalized advantages (mean 0, std 1 per group) -----
    advantages = torch.zeros_like(rewards)
    eps = 1e-8

    for b in range(batch_size):
        start = b * group_size
        end = start + group_size
        group_rewards = rewards[start:end]
        mean_r = group_rewards.mean()
        std_r = group_rewards.std(unbiased=False)
        if std_r < eps:
            group_adv = torch.zeros_like(group_rewards)
        else:
            group_adv = (group_rewards - mean_r) / (std_r + eps)
        advantages[start:end] = group_adv

    # ----- 6. Optional KL penalty vs frozen reference model -----
    kl_loss = torch.tensor(0.0, device=device)
    if kl_coef > 0.0 and ref_model is not None:
        with torch.no_grad():
            ref_outputs = ref_model(sequences, attention_mask=non_pad_mask)
            ref_logits = ref_outputs.logits
            ref_log_probs_full = F.log_softmax(ref_logits, dim=-1)
            ref_token_logprobs = ref_log_probs_full[:, :-1, :].gather(
                dim=-1,
                index=sequences[:, 1:].unsqueeze(-1),
            ).squeeze(-1)  # [B * G, T - 1]

        kl_per_seq = []
        for i in range(n_sequences):
            L_prompt = int(expanded_prompt_lengths[i].item())
            T_i = int(seq_lens[i].item())
            if T_i <= L_prompt:
                kl_seq = ref_token_logprobs[i, :0].sum()
            else:
                start = L_prompt - 1
                end = T_i - 1
                lp_pi = token_logprobs[i, start:end]
                lp_ref = ref_token_logprobs[i, start:end]
                # Sample-based per-token KL ≈ mean(log π - log π_ref)
                kl_seq = (lp_pi - lp_ref).mean()
            kl_per_seq.append(kl_seq)
        kl_per_seq = torch.stack(kl_per_seq, dim=0)
        kl_loss = kl_per_seq.mean()

    # ----- 7. Policy gradient loss (and KL term if enabled) -----
    # Detach advantages so gradients flow only through logprobs.
    policy_loss = -(advantages.detach() * seq_logprobs).mean()
    loss = policy_loss + kl_coef * kl_loss

    return {
        "loss": loss,
        "advantages": advantages,
        "rewards": rewards,
        "logprobs": seq_logprobs,
        "responses": responses,
        "prompts": expanded_prompts,
    }


# ---------- Sanity run on one batch (no optimizer step) ----------

demo_batch_prompts = [ex["prompt"] for ex in train_data[:2]]  # small batch
demo_out = grpo_step(demo_batch_prompts, group_size=GROUP_SIZE)

print("Demo batch size:", len(demo_batch_prompts))
print("Group size:", GROUP_SIZE)
print("Total sequences:", len(demo_out["responses"]))
print("Advantages shape:", demo_out["advantages"].shape)
print("Rewards shape:", demo_out["rewards"].shape)
print("Loss:", demo_out["loss"].item())

print("\nSample responses and rewards:")
for i in range(min(4, len(demo_out["responses"]))):
    print(f"Prompt:   {demo_out['prompts'][i]}")
    print(f"Response: {demo_out['responses'][i]!r}")
    print(f"Reward:   {float(demo_out['rewards'][i].item())}")
    print("---")

Demo batch size: 2
Group size: 4
Total sequences: 8
Advantages shape: torch.Size([8])
Rewards shape: torch.Size([8])
Loss: -2.32954740524292

Sample responses and rewards:
Prompt:   Add the numbers: 1 + 0 =
Response: '0 ; + 0 ='
Reward:   0.0
---
Prompt:   Add the numbers: 1 + 0 =
Response: '6 − 15 + 1'
Reward:   0.0
---
Prompt:   Add the numbers: 1 + 0 =
Response: '1 + 1 + 1'
Reward:   1.0
---
Prompt:   Add the numbers: 1 + 0 =
Response: '1 + 2 + 0'
Reward:   1.0
---


In [5]:
# Cell 6: Training Loop

from torch.optim import AdamW

# Hyperparameters for GRPO training
TRAIN_STEPS   = 200       # total update steps
BATCH_SIZE    = 16        # number of prompts per GRPO step
LEARNING_RATE = 1e-5
LOG_INTERVAL  = 20

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Precompute list of all training prompts
train_prompts = [ex["prompt"] for ex in train_data]

print(f"Number of training prompts: {len(train_prompts)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training steps: {TRAIN_STEPS}")
print("-" * 40)

running_loss = 0.0
running_reward = 0.0
running_count = 0

for step in range(1, TRAIN_STEPS + 1):
    # ---- Sample a mini-batch of prompts ----
    batch_prompts = random.sample(train_prompts, BATCH_SIZE)

    # ---- GRPO step: compute loss, rewards, etc. ----
    out = grpo_step(batch_prompts, group_size=GROUP_SIZE)

    loss = out["loss"]
    rewards = out["rewards"]   # shape: [BATCH_SIZE * GROUP_SIZE]

    # ---- Backpropagation & optimizer step ----
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # ---- Logging stats ----
    batch_mean_reward = rewards.mean().item()
    running_loss += loss.item()
    running_reward += batch_mean_reward
    running_count += 1

    if step % LOG_INTERVAL == 0 or step == 1:
        avg_loss = running_loss / running_count
        avg_reward = running_reward / running_count
        print(f"Step {step:4d}/{TRAIN_STEPS} "
              f"| avg loss: {avg_loss:.4f} "
              f"| avg reward: {avg_reward:.4f}")
        running_loss = 0.0
        running_reward = 0.0
        running_count = 0

print("Training loop finished.")

Number of training prompts: 1000
Batch size: 16
Training steps: 200
----------------------------------------
Step    1/200 | avg loss: 0.1686 | avg reward: 0.0312
Step   20/200 | avg loss: -0.0641 | avg reward: 0.0584
Step   40/200 | avg loss: -0.1971 | avg reward: 0.1203
Step   60/200 | avg loss: -0.3037 | avg reward: 0.1656
Step   80/200 | avg loss: -0.1988 | avg reward: 0.1852
Step  100/200 | avg loss: -0.0329 | avg reward: 0.2078
Step  120/200 | avg loss: -0.1530 | avg reward: 0.1992
Step  140/200 | avg loss: -0.1658 | avg reward: 0.2367
Step  160/200 | avg loss: -0.1341 | avg reward: 0.2938
Step  180/200 | avg loss: -0.2276 | avg reward: 0.2820
Step  200/200 | avg loss: -0.2367 | avg reward: 0.2828
Training loop finished.


In [6]:
# Cell 7: Evaluation and Generation

from statistics import mean

MAX_NEW_TOKENS_EVAL = 5
EVAL_BATCH_SIZE = 32


def generate_responses(model, prompts, max_new_tokens=MAX_NEW_TOKENS_EVAL):
    """
    Generate one completion per prompt (greedy decoding) and return the
    decoded responses (only the continuation, not the prompt).
    """
    model.eval()
    with torch.no_grad():
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(device)

        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]
        prompt_lengths = attention_mask.sum(dim=1)  # [B]

        gen_sequences = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,              # greedy for evaluation
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
        )

        sequences = gen_sequences
        non_pad_mask = sequences != tokenizer.pad_token_id
        seq_lens = non_pad_mask.sum(dim=1)

        responses = []
        for i in range(sequences.size(0)):
            L_prompt = int(prompt_lengths[i].item())
            T_i = int(seq_lens[i].item())
            gen_tokens = sequences[i, L_prompt:T_i]
            text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
            responses.append(text.strip())

    return responses


def evaluate_model(model, data, batch_size=EVAL_BATCH_SIZE):
    """
    Compute mean reward of `model` on a dataset (list of dicts with 'prompt').
    """
    prompts = [ex["prompt"] for ex in data]
    all_rewards = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start : start + batch_size]
        responses = generate_responses(model, batch_prompts)
        rewards = get_reward(batch_prompts, responses)  # [B] tensor on device
        all_rewards.extend(rewards.cpu().tolist())

    return mean(all_rewards)


# -------- Quantitative: Baseline vs Fine-tuned --------

print("Evaluating baseline (frozen reference model)...")
baseline_mean_reward = evaluate_model(ref_model, eval_data)
print(f"Baseline mean reward on eval set: {baseline_mean_reward:.4f}")

print("\nEvaluating fine-tuned model...")
finetuned_mean_reward = evaluate_model(model, eval_data)
print(f"Fine-tuned mean reward on eval set: {finetuned_mean_reward:.4f}")

improvement = finetuned_mean_reward - baseline_mean_reward
print(f"\nReward improvement: {improvement:.4f}")

# -------- Qualitative: 5 example generations (fine-tuned model) --------

print("\n=== Qualitative Examples (Fine-tuned Model) ===")

# Pick 5 fixed examples from eval_data for reproducibility
example_indices = list(range(min(5, len(eval_data))))
example_prompts = [eval_data[i]["prompt"] for i in example_indices]
example_targets = [eval_data[i]["target"] for i in example_indices]

example_responses = generate_responses(model, example_prompts)
example_rewards = get_reward(example_prompts, example_responses).cpu().tolist()

for i, (p, tgt, resp, rew) in enumerate(
    zip(example_prompts, example_targets, example_responses, example_rewards), start=1
):
    parsed = extract_first_int_from_text(resp)
    print(f"\nExample {i}")
    print(f"Prompt:   {p}")
    print(f"Target:   {tgt}")
    print(f"Response: {resp!r}")
    print(f"Parsed:   {parsed}")
    print(f"Reward:   {rew:.1f}")

Evaluating baseline (frozen reference model)...
Baseline mean reward on eval set: 0.0150

Evaluating fine-tuned model...
Fine-tuned mean reward on eval set: 0.2500

Reward improvement: 0.2350

=== Qualitative Examples (Fine-tuned Model) ===

Example 1
Prompt:   Add the numbers: 9 + 4 =
Target:   13
Response: '4 = 4 = 4'
Parsed:   4
Reward:   0.0

Example 2
Prompt:   Add the numbers: 3 + 9 =
Target:   12
Response: '10 = 11 = 12'
Parsed:   10
Reward:   0.0

Example 3
Prompt:   Add the numbers: 6 + 9 =
Target:   15
Response: '10 = 11 = 12'
Parsed:   10
Reward:   0.0

Example 4
Prompt:   Add the numbers: 0 + 7 =
Target:   7
Response: '7 = 8 = 9'
Parsed:   7
Reward:   1.0

Example 5
Prompt:   Add the numbers: 8 + 4 =
Target:   12
Response: '9 = 10 = 11'
Parsed:   9
Reward:   0.0


### **Cell 8: Analysis**

**Success**

- The baseline (frozen reference) model achieved a very low mean reward on the evaluation set (around 0.02).
- After GRPO fine-tuning, the mean reward increased to about 0.25 on the same evaluation set.
- The training logs also showed a steady increase in average batch reward across steps.
- This indicates that the model did learn to perform the addition task better than the baseline, although performance is still far from perfect.

**Dynamics and reward behavior**

- The fine-tuned model often produces outputs of the form `"7 = 8 = 9"` or `"9 = 10 = 11"` rather than a single clean answer.
- Because the reward function only checks the **first integer** in the response, outputs like `"7 = 8 = 9"` receive full reward if the first number is correct.
- This behavior is not full “reward hacking,” but it shows a mismatch between the reward function and the ideal behavior (a single, clean sum).
- A stricter reward (for example, requiring the entire string to match the exact correct answer) would likely reduce this pattern and push the model toward more precise outputs.

**Effect of group size 𝐺**

- In this experiment, the group size was set to \(G = 4\), so each prompt had 4 sampled completions used to compute group-normalized advantages.
- Increasing \(G\):
  - Pros: Better estimation of the reward distribution per prompt, lower variance advantages, and potentially more stable learning signals.
  - Cons: Higher computational cost per step and fewer parameter updates per unit time for a fixed budget.
- Decreasing \(G\):
  - Pros: Cheaper updates and more frequent optimizer steps.
  - Cons: Noisier advantage estimates and more unstable training, since each group’s rewards are based on very few samples.
- In practice, there is a trade-off: a moderate \(G\) (like 4–8) often balances stability and compute cost for small-scale experiments like this one.